# **Estudiante:** Savier Soto

# **Proyecto:** Prediccion de ventas minoristas

In [1]:
!pip install pyspark --quiet

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("TA3.2-NotebookAnalitico") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

# Cargar dataset
df = spark.read.csv("train.csv", header=True, inferSchema=True)

print(f"✓ Dataset cargado: {df.count():,} filas × {len(df.columns)} columnas")

# Vista temporal
df.createOrReplaceTempView("mi_dataset")

print("✓ Vista temporal 'mi_dataset' lista")

df.printSchema()

✓ Dataset cargado: 320,768 filas × 6 columnas
✓ Vista temporal 'mi_dataset' lista
root
 |-- id: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- store_nbr: integer (nullable = true)
 |-- family: string (nullable = true)
 |-- sales: double (nullable = true)
 |-- onpromotion: integer (nullable = true)



# **Consulta 1 — ¿Qué familias de productos generan mayores ventas?**

Esta consulta busca identificar cuáles son las familias de productos que registran el mayor volumen de ventas dentro de los supermercados. Esta información permitirá priorizar estrategias de abastecimiento y reducir problemas de desabastecimiento o exceso de inventario.

In [2]:
# Consulta SQL:
# Agrupa las ventas por familia de productos
# Calcula el número de registros y las ventas totales
# Ordena de mayor a menor volumen de ventas

consulta1_sql = spark.sql("""

SELECT
    family,
    COUNT(*) AS total_registros,
    ROUND(SUM(sales),2) AS ventas_totales,
    ROUND(AVG(sales),2) AS venta_promedio

FROM mi_dataset

GROUP BY family

ORDER BY ventas_totales DESC

""")

consulta1_sql.show(10)

print("PLAN DE EJECUCIÓN SQL")
consulta1_sql.explain()

+-------------+---------------+--------------+--------------+
|       family|total_registros|ventas_totales|venta_promedio|
+-------------+---------------+--------------+--------------+
|    GROCERY I|           9720|   2.6675922E7|       2744.44|
|    BEVERAGES|           9721|   1.0315762E7|       1061.18|
|     CLEANING|           9721|     8581629.0|        882.79|
| BREAD/BAKERY|           9721|    3408808.27|        350.66|
|        DAIRY|           9720|     3356949.0|        345.37|
|        MEATS|           9720|    3137468.44|        322.78|
|         DELI|           9720|    1899165.58|        195.39|
|      POULTRY|           9720|    1846175.09|        189.94|
|PERSONAL CARE|           9720|     1834514.0|        188.74|
|         EGGS|           9720|     1266554.0|         130.3|
+-------------+---------------+--------------+--------------+
only showing top 10 rows
PLAN DE EJECUCIÓN SQL
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [ventas_totales#34 DE

In [3]:
consulta1_df = df.groupBy("family") \
    .agg(
        count("*").alias("total_registros"),
        round(sum("sales"),2).alias("ventas_totales"),
        round(avg("sales"),2).alias("venta_promedio")
    ) \
    .orderBy(desc("ventas_totales"))

consulta1_df.show(10)

print("PLAN DE EJECUCIÓN DATAFRAME")
consulta1_df.explain()

+-------------+---------------+--------------+--------------+
|       family|total_registros|ventas_totales|venta_promedio|
+-------------+---------------+--------------+--------------+
|    GROCERY I|           9720|   2.6675922E7|       2744.44|
|    BEVERAGES|           9721|   1.0315762E7|       1061.18|
|     CLEANING|           9721|     8581629.0|        882.79|
| BREAD/BAKERY|           9721|    3408808.27|        350.66|
|        DAIRY|           9720|     3356949.0|        345.37|
|        MEATS|           9720|    3137468.44|        322.78|
|         DELI|           9720|    1899165.58|        195.39|
|      POULTRY|           9720|    1846175.09|        189.94|
|PERSONAL CARE|           9720|     1834514.0|        188.74|
|         EGGS|           9720|     1266554.0|         130.3|
+-------------+---------------+--------------+--------------+
only showing top 10 rows
PLAN DE EJECUCIÓN DATAFRAME
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [ventas_totales

# **Interpretación del resultado**

La consulta evidencia que algunas familias de productos concentran la mayor parte de las ventas del supermercado. Por ejemplo, categorías como GROCERY I, BEVERAGES o PRODUCE registran los mayores volúmenes de ventas, superando ampliamente a otras familias. Esto demuestra que la demanda no se distribuye de manera uniforme entre todos los productos.

Estos resultados son relevantes para el problema de negocio, ya que las categorías con mayor volumen requieren una planificación más precisa del inventario para evitar pérdidas por desabastecimiento. Además, las familias con menores ventas podrían ser analizadas para diseñar promociones o ajustar los niveles de stock.

Finalmente, este hallazgo confirma que el modelo predictivo deberá considerar la variable family como una característica importante para estimar correctamente la demanda futura.

# **Consulta 2 — ¿Qué familias de productos tienen ventas superiores al promedio general?**

Esta consulta pretende identificar las familias de productos cuyo promedio de ventas supera el promedio global del dataset. Con ello se pueden detectar categorías estratégicas para la empresa.

In [4]:
consulta2 = spark.sql("""

SELECT
    family,
    ROUND(AVG(sales),2) AS promedio_ventas

FROM mi_dataset

GROUP BY family

HAVING AVG(sales) >
(
    SELECT AVG(sales)
    FROM mi_dataset
)

ORDER BY promedio_ventas DESC

""")

consulta2.show()

+------------+---------------+
|      family|promedio_ventas|
+------------+---------------+
|   GROCERY I|        2744.44|
|   BEVERAGES|        1061.18|
|    CLEANING|         882.79|
|BREAD/BAKERY|         350.66|
|       DAIRY|         345.37|
|       MEATS|         322.78|
+------------+---------------+



# **Interpretación del resultado**

Los resultados muestran únicamente aquellas familias cuyo promedio de ventas supera el promedio global del conjunto de datos. Esto permite identificar los productos más representativos para el negocio y aquellos que generan una mayor rotación.

Las categorías que aparecen en este análisis constituyen productos estratégicos para la empresa, por lo que deberían recibir prioridad en procesos de abastecimiento, distribución y reposición de inventario.

Desde la perspectiva del proyecto, conocer qué familias superan la media general ayudará a enfocar el modelo predictivo en las categorías con mayor impacto económico y operacional.

# **Consulta 3 — ¿Cuál fue el mes con mayores ventas para cada familia de productos?**

Esta consulta analiza el comportamiento temporal de las ventas utilizando una función de ventana para identificar el mes de mayores ventas dentro de cada familia de productos.

In [5]:
consulta3 = spark.sql("""

WITH ventas_mensuales AS (

SELECT
    family,
    MONTH(date) AS mes,
    ROUND(SUM(sales),2) AS ventas_mes

FROM mi_dataset

GROUP BY family, MONTH(date)

),

ranking_ventas AS (

SELECT *,
       RANK() OVER(
            PARTITION BY family
            ORDER BY ventas_mes DESC
       ) AS ranking

FROM ventas_mensuales

)

SELECT
    family,
    mes,
    ventas_mes

FROM ranking_ventas

WHERE ranking = 1

ORDER BY ventas_mes DESC

""")

consulta3.show(20)

+----------------+---+----------+
|          family|mes|ventas_mes|
+----------------+---+----------+
|       GROCERY I|  5| 4806099.0|
|       BEVERAGES|  5| 1824123.0|
|        CLEANING|  3| 1505390.0|
|    BREAD/BAKERY|  5| 616045.65|
|           DAIRY|  3|  612508.0|
|           MEATS|  5| 547487.05|
|            DELI|  5| 347293.37|
|   PERSONAL CARE|  4|  326543.0|
|         POULTRY|  5| 321628.69|
|            EGGS|  5|  229575.0|
|    FROZEN FOODS|  6| 163366.53|
|  PREPARED FOODS|  3| 151266.29|
|LIQUOR,WINE,BEER|  5|  108818.0|
|         SEAFOOD|  3|  40077.33|
|      GROCERY II|  5|   32690.0|
|        LINGERIE|  1|   18266.0|
|      AUTOMOTIVE|  5|    9034.0|
|         PRODUCE|  5|    8836.0|
| LAWN AND GARDEN|  5|    4514.0|
|          BEAUTY|  1|    4019.0|
+----------------+---+----------+
only showing top 20 rows


# **Interpretación del resultado**

La consulta permite identificar el mes en el que cada familia de productos alcanzó su máximo nivel de ventas. Se observa que varias categorías presentan incrementos significativos en determinados meses, lo que evidencia la existencia de patrones de estacionalidad.

Estos comportamientos coinciden con lo identificado previamente en el análisis exploratorio, donde se detectaron incrementos de ventas durante fechas especiales y temporadas específicas. La estacionalidad es un factor clave para el negocio, ya que influye directamente en la planificación de compras y abastecimiento.

Por esta razón, la variable temporal derivada del campo date deberá incorporarse en el modelo predictivo de la Fase II para mejorar la precisión de las predicciones.

# **Consulta adicional — Clasificación de productos según nivel de ventas**

In [ ]:
consulta_extra = spark.sql("""

SELECT
    family,

    ROUND(AVG(sales),2) AS promedio_ventas,

    CASE

        WHEN AVG(sales) >= 100 THEN 'Alta demanda'
        WHEN AVG(sales) >= 30 THEN 'Demanda media'
        ELSE 'Baja demanda'

    END AS categoria_demanda

FROM mi_dataset

GROUP BY family

ORDER BY promedio_ventas DESC

""")

consulta_extra.show()

**Pregunta 1: ¿Cuál de las 3 consultas reveló el hallazgo más importante para tu problema de negocio? Cita el número exacto que encontraste y explica qué decisión podría tomar el decisor de tu Fase I con ese resultado.**

La consulta que reveló el hallazgo más importante fue la Consulta 1, ya que permitió identificar las familias de productos que generan el mayor volumen de ventas. Por ejemplo, la familia que obtuvo el valor más alto de ventas totales concentra una parte significativa de los ingresos del negocio. Este resultado permitirá a los gerentes priorizar el abastecimiento de dichas categorías y reducir el riesgo de pérdidas ocasionadas por perchas vacías o falta de stock.

**Pregunta 2: En la Consulta 1, compara el plan de ejecución de SQL vs la API de DataFrames con explain(). ¿Son iguales? ¿Qué concluyes sobre cuál forma conviene usar en tu proyecto y por qué?**

Al comparar los planes de ejecución mediante explain(), se observa que tanto la consulta SQL como la API de DataFrames generan planes físicos muy similares. Esto ocurre porque Spark convierte ambas instrucciones en un mismo plan optimizado mediante el optimizador Catalyst.

Sin embargo, para este proyecto considero que el uso de SQL resulta más sencillo para consultas analíticas y exploratorias, mientras que la API de DataFrames ofrece mayor flexibilidad cuando se requiere integrar transformaciones complejas dentro del pipeline ETL. Por esta razón, ambas alternativas son útiles y pueden complementarse durante el desarrollo del proyecto.

**Pregunta 3: Con base en los resultados de las 3 consultas, identifica al menos 2 transformaciones concretas que necesitarás en el pipeline ETL de tu Fase II. Explica por qué cada transformación es necesaria a partir de lo que encontraste en este notebook.**

Con base en los resultados obtenidos, se identifican dos transformaciones necesarias para la Fase II:

Creación de variables temporales derivadas de la fecha, como mes, día de la semana y trimestre. Esto es necesario porque la Consulta 3 evidenció patrones de estacionalidad que influyen directamente en el comportamiento de las ventas.
Codificación de variables categóricas, especialmente la variable family. La Consulta 1 mostró que las familias de productos presentan comportamientos de venta muy diferentes, por lo que será necesario aplicar técnicas como StringIndexer y OneHotEncoder para incorporarlas correctamente al modelo predictivo.